In [1]:
import os  # Operating system utilities
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # Set CUDA device visibility (disable GPU)

import pennylane as qml  # PennyLane for quantum machine learning
import torch  # PyTorch library for deep learning
import torch.optim as optim  # Optimizers for gradient-based optimization
import torch.nn as nn  # Neural network modules and loss functions
import copy  # Used for creating deep copies of objects
import time  # Provides time-related functionality
import deepxde as dde  # DeepXDE for solving differential equations
import numpy as np  # NumPy library for numerical computations
import matplotlib.pyplot as plt  # Matplotlib for plotting
from torch.utils.data import DataLoader  # DataLoader for handling data batches

# Set the default backend for DeepXDE to PyTorch
dde.backend.set_default_backend("pytorch")

# Define the default data type for PyTorch tensors
dtype = torch.float

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


Setting the default backend to "pytorch". You can change it in the ~/.deepxde/config.json file or export the DDE_BACKEND environment variable. Valid options are: tensorflow.compat.v1, tensorflow, pytorch, jax, paddle (all lowercase)


In [2]:
def Solution(x, y, params):
    """
    Computes the exact values of wave function.

    Args:
        x (torch.Tensor): Space coordinates in the X direction.
        y (torch.Tensor): Space coordinates in the Y direction.
        params (list): List containing box length, box width, pi, quantum number (x-direction), quantum number (x-direction), planck constant,  particle mass.

    Returns:
        torch.Tensor: wave function values respect to quantum numbers.
    """
    Lx, Ly, w, l, m, h_bar, mass = params
    return (2/np.sqrt(Lx*Ly))*torch.sin((l*torch.pi/Lx)*x)*torch.sin((m*torch.pi/Ly)*y)

def E_func(params):
    """
    Computes the exact value of energy level.

    Args:
        params (list): List containing box length, box width, pi, quantum number (x-direction), quantum number (x-direction), planck constant,  particle mass.

    Returns:
        torch.Tensor: energy level value respect to quantum numbers.
    """
    Lx, Ly, w, l, m, h_bar, mass = params
    return (1/(2*mass))*((h_bar*torch.pi)**2)*((pow(l, 2)/Lx**2) + (pow(m, 2)/Ly**2))

def phy_loss(psi, psi_xx, psi_yy, E, params):
    """
    Computes the physics-informed loss function (physics experssion).

    Args:
        psi (torch.Tensor): Wave function predicted by neural network.
        psi_xx (torch.Tensor): Second derivative of the predicted wave function with respect to x.
        psi_yy (torch.Tensor): Second derivative of the predicted wave function with respect to y.
        E (torch.Tensor): The energy level predicted by the neural network.
        params (list): List containing box length, box width, pi, quantum number (x-direction), quantum number (x-direction), planck constant,  particle mass.

    Returns:
        torch.Tensor: Mean of physics-informed loss function.
    """
    Lx, Ly, w, l, m, h_bar, mass = params
    return torch.mean(((pow(h_bar, 2)/mass)*(psi_xx + psi_yy) + 2*(E)*psi)**2)

def norm_loss(psi, diff):
    """
    Computes the normalization loss function (normalization experssion).

    Args:
        psi (torch.Tensor): Wave function predicted by neural network.
        diff (list): List of differential elements [dx, dy].

    Returns:
        torch.Tensor: Mean of normalization loss function.
    """
    dx, dy = diff
    return torch.abs(1 - torch.sqrt(torch.sum((psi**2)*dx*dy)))/torch.sqrt(torch.sum((psi**2)*dx*dy))

def train_PINN(model, op_params, loss_weights, batches, domain, params , num_itr, energy_level, seed, save_model=[False, "model"]):
    """
    Trains a physics-informed neural network (PINN) model.

    Args:
        model: The neural network model.
        op_params (tuple): Tuple containing betas, eps, and lr.
        loss_weights (list): List containing weights of loss function.
        batches: Number of points in the domain.
        domain (list): Defined domain in the interval [0, 2] for x and y axis.
        params (list): List containing box length, box width, pi, quantum number (x-direction), quantum number (x-direction), planck constant,  particle mass.
        num_itr (int): Number of iteration.
        energy_level (int): The number of energy levels to be calculated.
        save_model (list): Optional flag for saving the model.

    Returns:
        Trained models for each energy level, Loss functions and their components.
    """
    # Set the random seed for reproducibility
    torch.manual_seed(seed)
    # Initialize an empty list for neural networks
    bNNs = list()
    
    # Unpack optimizer parameters: betas, eps, and learning rate (lr)
    betas, eps , lr = op_params
    
    # Initialize empty list for different losses history to the respective energy
    loss_histories = []
    
    # Generate x and y points
    x = torch.linspace(domain[0][0], domain[0][1], batches[0])
    y = torch.linspace(domain[1][0], domain[1][1], batches[1])
    
    # Make differential x and y elements
    dx, dy = x[1]-x[0], y[1]-y[0]
    diff = [dx, dy]

    # Create 2D grids (X and Y) using meshgrid with "ij" indexing
    X, Y = torch.meshgrid(x, y, indexing="ij")

    # Record the start time before training begins
    time_i = time.time()
    
    # Preprocess data: stack X and Y, shuffle rows
    data = torch.vstack([X.ravel(), Y.ravel()]).T
    indexes = torch.randperm(data.shape[0])
    data = data[indexes]
    
    # Enable gradient tracking for the 'data' tensor
    data.requires_grad = True
    
    # Set the value of total energy and shooting energy changes equal to 1
    Et, dE = torch.tensor([0.0]), 0.0

    # Training loop over number of energy level
    for el in range(energy_level):
        # If the energy level (el) is greater than 0, increase the number of iterations by 1000
        if el > 0 :
            dE = 1.0 # Set the value of shooting energy changes equal to 1
            num_itr+=1000
            
        # Initialize the best trained neural network model (bNN) as None
        bNN = None
        # Initialize total loss (tot_l) as None
        tot_l = None
        
        # Initialize empty lists for different loss histories
        phy_loss_history = [] # physics-informed loss
        reg_loss_history = [] # normalization loss function
        overall_loss_history = [] # Overall loss function history
        energy_loss_history = [] # predicted energy from neural network
        
        # Initialize an upper limit for loss (l_lim) with a large value
        l_lim = 1e+20
        
        # Reset model parameters and update the optimizer
        model.set_net_params()
        model.set_energy_params(1.0)
        optimizer = optim.Adam(model.parameters(), betas=betas, eps=eps, lr=lr)
        
        # set the previous energy level+dE and shooting energy
        Ep, Es = Et.detach() + dE, 1.0
        
        # Training loop over iterations
        for itr in range(num_itr):
            # Clear the gradients of the DDE loss with respect to model parameters
            dde.grad.clear()
            
            # Pass the input data (features) through the neural network and obtain wave function and energy level
            psi_pred , E = model(data)
            Et = E + Ep # The energy level is raised to the energy level of the previous level
                
            # Shoot the energy level to the value of Es
            if (itr+1)%500==0 and Et < (Ep+0.5):
                Es = Es + dE
                model.set_energy_params(Es)

            # Compute physics-informed and normalization losses, then combine them
            psi_xx = dde.grad.hessian(psi_pred, data, i=0, j=0) # Hessian along x-axis
            psi_yy = dde.grad.hessian(psi_pred, data, i=1, j=1) # Hessian along y-axis
            phy_l = phy_loss(psi_pred, psi_xx, psi_yy, Et, params)
            reg_l = norm_loss(psi_pred, diff)
            tot_l = loss_weights[0]*phy_l + loss_weights[1]*reg_l

            # Update model parameters: zero gradients, compute backward pass, and perform optimization step
            optimizer.zero_grad()
            tot_l.backward(retain_graph=True)
            optimizer.step()

            # Append each overall loss function and its expressions to the respective lists
            phy_loss_history.append(phy_l.item())
            reg_loss_history.append(reg_l.item())
            energy_loss_history.append(Et.item())
            overall_loss_history.append(tot_l.item())

            # Print the loss value at specific iteration during training
            if (itr+1)%500==0:
                print(f'iteration {itr+1}/{num_itr}, loss = {tot_l}')

            # Update the best neural network if the total loss is lower than the current limit
            if  tot_l < l_lim:
                bNN =  copy.deepcopy(model)
                l_lim = tot_l

            # Clean up memory by deleting variables tot_l, psi_pred, and E
            del tot_l, psi_pred, E

        # Collect different and neural networks and loss histories for analysis
        bNNs.append(bNN)
        loss_histories.append([overall_loss_history, phy_loss_history, reg_loss_history, energy_loss_history, Ep])

        # Find the minimum overall loss value, its iteration, and print the result
        min_loss = min(overall_loss_history)
        min_index_loss = overall_loss_history.index(min_loss)
        print(f"The minimum overall loss function value occurs at epoch {min_index_loss+1}: {overall_loss_history[min_index_loss]}")

    # Measure the runtime of the training process
    time_f = time.time()
    runtime = time_f - time_i
    print(f"train took {runtime} s")

    # Save model state dictionaries and loss histories if requested
    if save_model[0] == True:
        for el,m in enumerate(bNNs):
            torch.save(m.state_dict(), save_model[1]+str(el)+".pth")
        torch.save(loss_histories, "loss_"+save_model[1]+".pt")

    return bNNs, loss_histories

def compute_qm_solutions(domain, num_test):
    """
    Compute the exact solutions for quantum mechanics models.

    Args:
        domain (list): Defined domain in the interval [0, 2] for x and y axis.
        num_test (int): Number of test data points.

    Returns:
        tuple: data points, exact energy levels list, and wavefunctions list.
    """
    # Generate x and y points
    x = torch.linspace(domain[0][0], domain[0][1], num_test)
    y = torch.linspace(domain[1][0], domain[1][1], num_test)

    # Generate x and y points
    X, Y = torch.meshgrid(x, y, indexing="ij")
    data = torch.vstack([X.ravel(), Y.ravel()]).T

    psi_exact_list = []
    E_exact_list = []
    degeneracy_psi = []

    # First configuration (first energy level)
    Lx, Ly, w, l, m, h_bar, mass = (domain[0][1]-domain[0][0]), (domain[1][1]-domain[1][0]) , torch.pi, 1, 1, 1, 1
    params = [Lx, Ly, w, l, m, h_bar, mass]
    psi_exact_list.append([Solution(data[:, 0], data[:, 1], params)])
    E_exact_list.append(E_func(params))

    # Degeneracy configurations (second energy level)
    Lx, Ly, w, l, m, h_bar, mass = (domain[0][1]-domain[0][0]), (domain[1][1]-domain[1][0]) , torch.pi, 1, 2, 1, 1
    params = [Lx, Ly, w, l, m, h_bar, mass]
    degeneracy_psi.append(Solution(data[:, 0], data[:, 1], params))
    Lx, Ly, w, l, m, h_bar, mass = (domain[0][1]-domain[0][0]), (domain[1][1]-domain[1][0]) , torch.pi, 2, 1, 1, 1
    params = [Lx, Ly, w, l, m, h_bar, mass]
    degeneracy_psi.append(Solution(data[:, 0], data[:, 1], params))
    psi_exact_list.append(degeneracy_psi)
    E_exact_list.append(E_func(params))

    # Final configuration (third energy level)
    Lx, Ly, w, l, m, h_bar, mass = (domain[0][1]-domain[0][0]), (domain[1][1]-domain[1][0]) , torch.pi, 2, 2, 1, 1
    params = [Lx, Ly, w, l, m, h_bar, mass]
    psi_exact_list.append([Solution(data[:, 0], data[:, 1], params)])
    E_exact_list.append(E_func(params))

    return X, Y, data, E_exact_list, psi_exact_list

In [3]:
class CNeuralNet(nn.Module):
    """
    Classical neural network class.

    Args:
        input_size (int): Size of the input features.
        hidden_size (int): Number of hidden units in the neural network.
        output_size (int): Size of the output.
        domain (list): Defined domain in the interval [0, 2] for x and y axis.

    Attributes:
        fnn (nn.Sequential): Feedforward neural network with two linear layer followed by a Tanh activation.

    Notes:
        - The classical neural network architecture consists of one linear layer for energy and two hidden layers for wave function.
        - The input layer has 'input_size' neurons.
        - The hidden layers for predicting wave function has 'hidden_size' neurons with a Tanh activation.
        - The output layer has 'output_size' neurons.
    """
    def __init__(self, input_size, hidden_size, output_size, domain, seed):
        super(CNeuralNet, self).__init__()
        self.seed = seed
        # Define the feedforward (1, 1) layer
        self.Energy = nn.Linear(1, 1)
        # Define the feedforward neural network (fnn)
        self.fnn = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, output_size))

        # Set the x-borders and y-borders based on the specified domain
        self.xborders = domain[0]
        self.yborders = domain[1]

    def forward(self, data):
        """
        Forward pass through the neural network.

        Args:
            data (torch.Tensor): Input data.

        Returns:
            torch.Tensor: Outputs predictions from the neural network.
        """
        # Extract input features
        x, y= data[:, 0].view(-1, 1), data[:, 1].view(-1, 1)
        
        # set lambda to pass through the linear layer (energy layer)
        lbd = torch.tensor([[1.0]])
        Energy = self.Energy(lbd)
        
        # Pass the data through the feedforward neural network (self.fnn) and combine outputs
        out = self.fnn(data)
        out1 = out[:, 0].view(-1, 1)
        out2 = out[:, 1].view(-1, 1)
        out = (1-torch.exp(-x+self.xborders[0]))*(1-torch.exp((x-self.xborders[1])))*out1*(1-torch.exp(-y+self.yborders[0]))*(1-torch.exp((y-self.yborders[1])))*out2

        return out, torch.abs(Energy)

    def init_weights(self, m):
        """
        Initializes weights and biases for a linear layer.

        Args:
            m (nn.Linear): The linear layer to initialize.

        Notes:
            - Applies uniform weight initialization within the range [-n0, n0].
            - Initializes biases with a constant value of 0.01.
        """
        if isinstance(m, nn.Linear):
            in_sz = torch.tensor(m.in_features)
            out_sz = torch.tensor(m.out_features)
            n0 = torch.sqrt(6/(in_sz+out_sz))
            m.weight.data.uniform_(-n0, n0)
            m.bias.data.fill_(0.01)

    def init_energy(self, shooting):
        """
        Initializes weights and biases for a linear layer.

        Args:
            m (nn.Linear): The linear layer to initialize.

        Notes:
            - Applies shooting number.
            - Initializes biases with a constant value of 0.0.
        """
        def inner(m):
            if isinstance(m, nn.Linear):
                m.weight.data.fill_(shooting)
                m.bias.data.fill_(0.0)
        return inner

    def set_net_params(self):
        """
        set weights and biases for linear layers.

        Args:
            None.
        """
        # Set the random seed for reproducibility
        torch.manual_seed(self.seed)
        self.fnn.apply(self.init_weights)

    def set_energy_params(self, shooting):
        """
        set shooting number for a linear layer.

        Args:
            shooting (float): The value of shooting energy.
        """
        self.Energy.apply(self.init_energy(shooting))

In [4]:
for h_size in [10, 30, 50]:
    # Set the random seed for reproducibility
    seed = 42
    torch.manual_seed(seed)
    # Define input, hidden, and output sizes and specify the domain (range) for input data
    input_size , hidden_size, output_size = 2, h_size, 2
    domain = [[0, 2],[0, 2]]
    
    # Create a classical neuralnet class and print the model architecture
    model_classic = CNeuralNet(input_size, hidden_size, output_size, domain, seed)
    
    # Print the model architecture
    print(model_classic)
        
    # Set the necessary parameters for training and test
    Lx, Ly, w, l, m, h_bar, mass = (domain[0][1]-domain[0][0]), (domain[1][1]-domain[1][0]) ,torch.pi, 1, 1, 1, 1
    params = [Lx, Ly, w, l, m, h_bar, mass]
    op_params = ([0.9, 0.99], 1e-8, 0.002)
    energy_level = 3
    num_itr = 1000
    batches, loss_weights = [10, 10], [1, 1]
    
    # Initialize an empty list to store the loss history
    loss_hist_classic = []
    # Specify whether to save the model and the desired filename
    save_model = [True, f"Classical model_{seed}_{h_size}"]
    
    # Train the physics-informed neural network (PINN)
    bmodels_classic, loss_hist_classic = train_PINN(model_classic, op_params, loss_weights, batches, domain, params, num_itr, energy_level, seed, save_model)

CNeuralNet(
  (Energy): Linear(in_features=1, out_features=1, bias=True)
  (fnn): Sequential(
    (0): Linear(in_features=2, out_features=10, bias=True)
    (1): Tanh()
    (2): Linear(in_features=10, out_features=10, bias=True)
    (3): Tanh()
    (4): Linear(in_features=10, out_features=2, bias=True)
  )
)
iteration 500/1000, loss = 0.23989661037921906
iteration 1000/1000, loss = 0.04197367653250694
The minimum overall loss function value occurs at epoch 999: 0.03969705104827881
iteration 500/2000, loss = 0.9966214895248413
iteration 1000/2000, loss = 0.8117470741271973
iteration 1500/2000, loss = 0.06026814505457878
iteration 2000/2000, loss = 0.010720094665884972
The minimum overall loss function value occurs at epoch 1988: 0.006831571459770203
iteration 500/3000, loss = 3.9354798793792725
iteration 1000/3000, loss = 1.2839858531951904
iteration 1500/3000, loss = 1.5480650663375854
iteration 2000/3000, loss = 0.0901537835597992
iteration 2500/3000, loss = 0.031868454068899155
itera